# TP 5 – Explorer avant de prédire (35 min)

**Objectif** : trouver 3 faits dans les données qui guideront nos modèles de l'après-midi.

**Règle d'or** : Spark ne dessine pas. On **agrège** (`groupBy`) ou on **échantillonne** (`sample`) dans Spark,
puis `toPandas()` sur un petit résultat, puis Matplotlib / Seaborn.

**Réussi si** : 4 graphiques affichés et 3 observations écrites.

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
trips = (spark.read.parquet("data/trips.parquet")
         .withColumn("heure", F.hour("tpep_pickup_datetime"))
         .withColumn("jour_semaine", F.dayofweek("tpep_pickup_datetime"))    # 1 = dimanche, 2 = lundi … 7 = samedi
         .join(zones.select(F.col("LocationID").alias("PULocationID"), "Borough"), "PULocationID"))
trips.cache().count()

### 1. Distribution des montants payés (`total_amount`) : histogramme

Indice : créer une colonne `tranche` = `floor(total_amount / 5) * 5`, puis `groupBy("tranche").count()`, puis `sns.barplot` ; `plt.xticks(rotation=90)` pour lire les étiquettes.

In [ ]:
tranches = ...
# sns.barplot(data=tranches, x="tranche", y="count")

### 2. Tarif du compteur (`fare_amount`, sans pourboire ni taxes) selon la distance : nuage de points sur un **échantillon**

Indice : `trips.sample(fraction=0.03, seed=1)`, puis `plt.scatter`. Limiter les axes : 0–30 miles, 0–120 $. C'est `fare_amount` que nous prédirons au TP 6.

In [ ]:
ech = ...
# plt.scatter(ech.trip_distance, ech.fare_amount, s=5, alpha=0.3)

### 3. Courses par jour de semaine et par heure : heatmap

Indice : `groupBy("jour_semaine", "heure").count()`, `toPandas().pivot(index=..., columns=..., values=...)`, `sns.heatmap`.

Piège de lecture : janvier 2024 compte 5 lundis, mardis et mercredis, mais 4 des autres jours. Comparez les **heures** entre elles, pas les jours.

In [ ]:
...

### 4. Montant moyen par arrondissement de départ : barres

In [ ]:
...

### 5. Vos 3 observations utiles pour **prédire le montant d'une course**

Regardez bien le nuage de points : que signifie la ligne horizontale ?

1. …
2. …
3. …

### Bonus – le même nuage de points en Plotly (interactif)

`pip install plotly` puis `plotly.express.scatter(ech, x=..., y=...)`.

In [ ]:
...